Before running the file Upload all your data set on your goole drive in a zip format

In [1]:
#Mount our google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# #before running this please change the RUNTIME to GPU (Runtime -> Change runtime type -> set harware accelarotor as GPU)
# #download and unzip the data from google drive Colab environment
# from google_drive_downloader import GoogleDriveDownloader as gdd
# #use only file id of the link
# #Note: Below link is just an example, Not an actual link. Actual Links are in ReadMe file
# #https://drive.google.com/file/d/1ubvKLzBDe5i1acxgGUK6ObeNBYCKUS07/view?usp=sharing
# #https://drive.google.com/file/d/1qXoslbnOFbSgQDKvvQJw3ukCjQqB2ijy/view?usp=sharing
# url = '/content/drive/MyDrive/Celeb-DF-v2.zip'
# gdd.download_file_from_google_drive(file_id = url,dest_path='./data.zip',unzip=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [4]:
#To get the average frame count
import json
import glob
import numpy as np
import cv2
import copy
#change the path accordingly
video_files =  glob.glob('/content/drive/MyDrive/Celeb-DF-v2/Celeb-real/*.mp4')
video_files1 =  glob.glob('/content/drive/MyDrive/Celeb-DF-v2/Celeb-synthesis/*.mp4')
video_files += video_files1
frame_count = []
for video_file in video_files:
  cap = cv2.VideoCapture(video_file)
  if(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))<150):
    video_files.remove(video_file)
    continue
  frame_count.append(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("frames" , frame_count)
print("Total number of videos: " , len(frame_count))
print('Average frame per video:',np.mean(frame_count))

frames [328, 529, 303, 350, 425, 459, 371, 505, 458, 326, 383, 409, 469, 307, 322, 376, 276, 534, 519, 422, 520, 400, 451, 479, 464, 380, 361, 445, 516, 470, 412, 398, 499, 428, 372, 380, 350, 469, 303, 303, 534, 469, 459, 520, 479, 529, 459, 350, 469, 303, 326, 529, 520, 479, 534, 459, 464, 326, 529, 350, 534, 520, 350, 534, 520, 326, 459, 529, 303, 469, 479, 464]
Total number of videos:  72
Average frame per video: 431.18055555555554


In [5]:
# to extract frame
def frame_extract(path):
  vidObj = cv2.VideoCapture(path)
  success = 1
  while success:
      success, image = vidObj.read()
      if success:
          yield image
!pip3 install face_recognition
!mkdir '/content/drive/My Drive/Face_only_data'
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import face_recognition
from tqdm.autonotebook import tqdm
# process the frames
def create_face_videos(path_list,out_dir):
  already_present_count =  glob.glob(out_dir+'*.mp4')
  print("No of videos already present " , len(already_present_count))
  for path in tqdm(path_list):
    out_path = os.path.join(out_dir,path.split('/')[-1])
    file_exists = glob.glob(out_path)
    if(len(file_exists) != 0):
      print("File Already exists: " , out_path)
      continue
    frames = []
    flag = 0
    face_all = []
    frames1 = []
    out = cv2.VideoWriter(out_path,cv2.VideoWriter_fourcc('M','J','P','G'), 30, (112,112))
    for idx,frame in enumerate(frame_extract(path)):
      #if(idx % 3 == 0):
      if(idx <= 150):
        frames.append(frame)
        if(len(frames) == 4):
          faces = face_recognition.batch_face_locations(frames)
          for i,face in enumerate(faces):
            if(len(face) != 0):
              top,right,bottom,left = face[0]
            try:
              out.write(cv2.resize(frames[i][top:bottom,left:right,:],(112,112)))
            except:
              pass
          frames = []
    try:
      del top,right,bottom,left
    except:
      pass
    out.release()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566164 sha256=cafa84cf7bdc501188732355ef33268a8db2b6c67e7c69857fb151c2670e0947
  Stored in directory: /root/.cache/pip/wheels/7a/eb/cf/e9eced74122b679557f597bb7c8e4c739cfcac526db1fd523d
Successfully built face-recognition-models


<ipython-input-5-fb26f6640e24>:21: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [6]:
create_face_videos(video_files,'/content/drive/My Drive/Face_only_data/')

No of videos already present  0


  0%|          | 0/72 [00:00<?, ?it/s]